Step 1 – Extract only COVID vaccine reports from VAERS raw data years 2020-2025

In [ ]:
# This code filters out only 'COVID-19' records between 2020-2025
# and combines data from VAERSVAX, VAERSDATA, and VAERSSYMPTOMS

import pandas as pd
import os

# Define the folder where your raw CSVs are stored
data_path = "../data/raw"

# Years to loop through
years = [2020, 2021, 2022, 2023, 2024, 2025]

# Helper: read CSV with encoding fallbacks + safe VAERS_ID dtype
def read_csv_safely(path: str) -> pd.DataFrame:
    for enc in ("utf-8", "cp1252", "latin1"):
        try:
            df = pd.read_csv(
                path,
                encoding=enc,
                low_memory=False,
                dtype={"VAERS_ID": "string"}
            )
            print(f"  ✓ Loaded {os.path.basename(path)} with encoding='{enc}'")
            return df
        except Exception as e:
            last_error = e
    raise RuntimeError(f"Failed to read {path}. Last error: {last_error}")

# To collect all COVID-19 data
all_covid_data = []

for year in years:
    print(f"\nProcessing {year}...")

    # File paths (now inside data/raw/)
    vax_file = os.path.join(data_path, f"{year}VAERSVAX.csv")
    data_file = os.path.join(data_path, f"{year}VAERSDATA.csv")
    symptoms_file = os.path.join(data_path, f"{year}VAERSSYMPTOMS.csv")

    # Load files
    vax_df = read_csv_safely(vax_file)
    data_df = read_csv_safely(data_file)
    symptom_df = read_csv_safely(symptoms_file)

    # Normalize column names
    vax_df.columns = [c.upper() for c in vax_df.columns]
    data_df.columns = [c.upper() for c in data_df.columns]
    symptom_df.columns = [c.upper() for c in symptom_df.columns]

    # Filter COVID-19 records from VAERSVAX
    covid_vax_df = vax_df[vax_df["VAX_TYPE"].astype("string").str.upper() == "COVID19"]

    # Get the COVID VAERS_IDs
    covid_ids = covid_vax_df["VAERS_ID"]

    # Filter other datasets using COVID VAERS_IDs
    covid_data = data_df[data_df["VAERS_ID"].isin(covid_ids)]
    covid_symptoms = symptom_df[symptom_df["VAERS_ID"].isin(covid_ids)]

    # Merge data + symptoms
    merged_df = covid_data.merge(covid_symptoms, on="VAERS_ID", how="left")

    # Merge VAX_NAME from COVID VAERSVAX
    merged_df = merged_df.merge(
        covid_vax_df[["VAERS_ID", "VAX_NAME"]],
        on="VAERS_ID",
        how="left"
    )

    # Add year column
    merged_df["YEAR"] = year

    # Add to list
    all_covid_data.append(merged_df)
    print(f"  → Rows after merge for {year}: {len(merged_df):,}")

# Combine all years
final_df = pd.concat(all_covid_data, ignore_index=True)

# Save to CSV (in data/processed for clarity)

output_file = "../data/processed/combined_covid_vaers.csv"
final_df.to_csv(output_file, index=False)

print(f"\n✅ Combined COVID-19 VAERS data saved as '{output_file}'")
print(final_df.head(10).to_string(index=False))
print(f"\nTotal rows: {len(final_df):,}")



Processing 2020...
  ✓ Loaded 2020VAERSVAX.csv with encoding='utf-8'
  ✓ Loaded 2020VAERSDATA.csv with encoding='cp1252'
  ✓ Loaded 2020VAERSSYMPTOMS.csv with encoding='utf-8'
  → Rows after merge for 2020: 19,131

Processing 2021...
  ✓ Loaded 2021VAERSVAX.csv with encoding='cp1252'
  ✓ Loaded 2021VAERSDATA.csv with encoding='cp1252'
  ✓ Loaded 2021VAERSSYMPTOMS.csv with encoding='utf-8'
  → Rows after merge for 2021: 1,216,414

Processing 2022...
  ✓ Loaded 2022VAERSVAX.csv with encoding='cp1252'
  ✓ Loaded 2022VAERSDATA.csv with encoding='cp1252'
  ✓ Loaded 2022VAERSSYMPTOMS.csv with encoding='utf-8'
  → Rows after merge for 2022: 344,779

Processing 2023...
  ✓ Loaded 2023VAERSVAX.csv with encoding='cp1252'
  ✓ Loaded 2023VAERSDATA.csv with encoding='cp1252'
  ✓ Loaded 2023VAERSSYMPTOMS.csv with encoding='utf-8'
  → Rows after merge for 2023: 83,442

Processing 2024...
  ✓ Loaded 2024VAERSVAX.csv with encoding='cp1252'
  ✓ Loaded 2024VAERSDATA.csv with encoding='cp1252'
  ✓ Loaded